# Optimizing LLM-Powered Applications: Cost, Tokens & Performance

**A hands-on workshop.** You'll take a small, real AI *coding agent* and make
it cheaper and faster — then **prove** the improvement with real numbers, not
guesses.

Building an LLM app that *works* is the easy part. Making it **cost-effective,
fast, and production-ready** is the real job. In this notebook you will:

1. Clone a lightweight coding agent (built from scratch — no heavy frameworks).
2. Run it on a fixed set of tasks and record **tokens, cost, and speed**.
3. Turn on optimizations **one at a time** and watch the numbers drop.
4. **Stack** optimizations and measure the combined win.

Everything runs in the browser — no local Python or VS Code install needed.
The only thing you provide is **your own API key** — either an **OpenRouter**
key or an **Anthropic** key (both free to create). You pick the provider,
the key, and the model in one configuration cell near the top; the rest of
the notebook adapts automatically.

> Every number in this notebook is a **real** token count from the model's API
> response — never estimated. That's the whole point: optimizations are judged
> by evidence.

## How this notebook works

The agent is normally a terminal chat app (`uv run coding-agent`). Here we drive
the **exact same code** in-process, one prompt at a time, so we can capture the
precise tokens and cost of each run. Whenever you see a cell run the agent, the
terminal equivalent is shown in the notes.

**The plan:** run the same fixed prompts as a *baseline*, then re-run them with
each optimization enabled and compare. Because the task never changes, any drop
in tokens/cost is caused by the optimization — nothing else.

Run the cells **top to bottom.** Start with setup.

---
## Step 1 — Clone the agent

Downloads the public repository. Safe to re-run (it reuses an existing clone).

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shrijayan/coding-agent.git"
REPO_BRANCH = "main"

BASE_DIR = os.getcwd()                       # /content on Colab
REPO_DIR = os.path.join(BASE_DIR, "coding-agent")

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print("Cloning", REPO_URL, "...")
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, REPO_DIR],
        check=True,
    )
else:
    # Already cloned (e.g. re-running this cell in an existing Colab
    # session) - pull so a fast-moving repo's latest fixes actually land
    # here instead of silently running a stale checkout indefinitely.
    print("Repo already present at", REPO_DIR, "- pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)

# Make `import coding_agent` work without a full package install.
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("REPO_DIR =", REPO_DIR)

## Step 2 — Install dependencies

This notebook is a **thin client** for the repository: it installs whatever
the project itself declares, straight from the repo's `pyproject.toml` — it
never lists individual packages of its own. We change into the cloned repo
and install it in editable mode with the `notebook` extra (which adds the
table/chart libraries used below).

The upshot: when the repository's dependencies change, you just re-pull the
repo and re-run this cell — there is nothing to update in the notebook.

In [ ]:
import subprocess, sys

# Install the project (and its declared dependencies) from its own manifest.
# `.[notebook]` = the repo's runtime deps + the notebook-only extras
# (pandas/matplotlib) declared under [project.optional-dependencies] in
# pyproject.toml. cwd=REPO_DIR is the "change into the repository directory
# first" step. Nothing about which packages to install lives in this notebook.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=REPO_DIR,
    check=True,
)

print("Dependencies installed from", REPO_DIR + "/pyproject.toml")

## Step 3 — Configuration (the only cell you normally edit)

Everything below is driven by three values: **provider**, **API key**, and
**model**. Set them once here and the rest of the notebook adapts — you never
touch the implementation code to switch providers.

- **`PROVIDER`** — `"openrouter"` or `"anthropic"` (the design is
  provider-agnostic; new providers are one entry in the `PROVIDERS` registry).
- **`API_KEY`** — paste your key, *or* leave it blank to load it securely from
  a Colab secret (named after the provider's env var, e.g. `OPENROUTER_API_KEY`
  or `ANTHROPIC_API_KEY`), an exported environment variable, or a hidden prompt.
- **`MODEL`** — leave blank for the provider's default. For OpenRouter you can
  also use a tier preset: `"low"`, `"medium"`, or `"high"`.

| Provider   | Presets / default model                                              | Get a key |
|------------|----------------------------------------------------------------------|-----------|
| openrouter | low `google/gemma-3.4b` · medium `qwen/qwen3.7-flash` · high `deepseek/deepseek-v4-flash-0731` (default) | https://openrouter.ai/keys |
| anthropic  | `claude-sonnet-5`                                                     | https://console.anthropic.com/settings/keys |

In [ ]:
# ==================== EDIT ME ====================
PROVIDER = "openrouter"   # "openrouter" or "anthropic"
API_KEY  = ""             # blank -> Colab secret / env var / hidden prompt
MODEL    = ""             # blank -> provider default; OpenRouter: "low"/"medium"/"high" or a full slug
# =================================================

import os, getpass

# Provider-agnostic registry. Adding a provider later = one entry here
# (its API-key env var, default model, and any tier presets) — no notebook
# logic changes. The model presets mirror the repo's models.yaml.
PROVIDERS = {
    "openrouter": {
        "env_var": "OPENROUTER_API_KEY",
        "default_model": "deepseek/deepseek-v4-flash-0731",
        "presets": {
            "low": "google/gemma-3.4b",
            "medium": "qwen/qwen3.7-flash",
            "high": "deepseek/deepseek-v4-flash-0731",
        },
        "keys_url": "https://openrouter.ai/keys",
    },
    "anthropic": {
        "env_var": "ANTHROPIC_API_KEY",
        "default_model": "claude-sonnet-5",
        "presets": {},
        "keys_url": "https://console.anthropic.com/settings/keys",
    },
}

provider = PROVIDER.strip().lower()
assert provider in PROVIDERS, (
    f"Unknown PROVIDER {PROVIDER!r}. Choose one of: {', '.join(PROVIDERS)}."
)
pcfg = PROVIDERS[provider]

def _load_api_key(env_var):
    """Resolve the selected provider's key without ever printing it."""
    if API_KEY.strip():
        return API_KEY.strip(), "notebook config"
    try:
        from google.colab import userdata  # type: ignore
        val = userdata.get(env_var)
        if val:
            return val.strip(), f"Colab secret '{env_var}'"
    except Exception:
        pass
    if os.environ.get(env_var):
        return os.environ[env_var].strip(), f"environment '{env_var}'"
    return getpass.getpass(f"Paste your {provider} API key (hidden): ").strip(), "hidden prompt"

# Resolve the model: a preset name -> its slug, blank -> provider default,
# anything else -> used verbatim (a full model slug).
_choice = MODEL.strip()
if not _choice:
    model = pcfg["default_model"]
elif _choice.lower() in pcfg["presets"]:
    model = pcfg["presets"][_choice.lower()]
else:
    model = _choice

# Load + store ONLY the selected provider's key (the agent requires just that).
key, key_src = _load_api_key(pcfg["env_var"])
assert key, f"No API key for {provider}. Get one at {pcfg['keys_url']}."
os.environ[pcfg["env_var"]] = key

# Point the agent at the selected provider + model. These AGENT_* env vars are
# exactly the override mechanism the repo already supports (see .env.example),
# so the notebook stays a client: no implementation code changes to switch.
os.environ["AGENT_PROVIDER"] = provider
os.environ["AGENT_MODEL"] = model

# A shorter summary threshold than .env.example's default (10), tuned so
# this notebook's short DEMO_PROMPTS conversation actually triggers
# summarization for the demo (see Optimization 1 below). Set explicitly
# (not setdefault) so it survives the auto-resolver right below it.
os.environ["AGENT_SUMMARY_THRESHOLD_MESSAGES"] = "8"

# Every other behavioral knob the agent requires comes straight from the
# repo's own .env.example - one parse, not a hand-copied list. That list
# kept drifting out of sync every time a new optimization added a required
# setting (AGENT_LOOP_GUARD_*, then AGENT_CONTEXT_PRUNE_*, then
# AGENT_DEDUP_MIN_CHARS each broke this cell in turn) - reading the file
# directly means any future required knob just works, as long as it ships
# a default in .env.example (already this project's rule for every one).
# Blank lines (secrets, optional overrides with a code-level fallback) are
# skipped on purpose - there's no default to apply for those here.
with open(os.path.join(REPO_DIR, ".env.example")) as _f:
    for _line in _f:
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _key, _, _value = _line.partition("=")
        _key, _value = _key.strip(), _value.strip()
        if _value:
            os.environ.setdefault(_key, _value)

print(f"Provider : {provider}")
print(f"Model    : {model}")
print(f"API key  : loaded from {key_src} (length {len(key)})")

## Step 4 — Sanity check & connectivity

Loads the agent's config and sends a tiny 1-word request to the model. If your
key is wrong, or the selected model slug isn't valid for your provider, this
fails **here** — loudly — instead of halfway through a demo.

In [ ]:
from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.models_config import read_models_yaml
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS

cfg = Config.from_env()
pricing = PricingTable.load()
pricing.require(cfg.model)

print("Base model      :", f"{cfg.provider} / {cfg.model}")
print("Session cost cap :", cfg.session_cost_cap_usd, "USD")
print("Optimizations available:", ", ".join(AVAILABLE_OPTIMIZATIONS) or "none")

# Live ping through the agent's own client.
from coding_agent.llm.factory import build_llm_client
from coding_agent.llm.messages import Message, TextPart

_client = build_llm_client(cfg)
try:
    _r = _client.send(
        system="Reply with the single word: ok",
        messages=[Message(role="user", parts=[TextPart("ping")])],
        tools=[],
    )
    print("\nConnectivity OK — model replied:", repr(_r.text.strip()[:60]))
    print("Reported usage :", _r.usage)
except Exception as e:
    print("\nConnectivity FAILED:", type(e).__name__, "-", str(e)[:300])
    print("Check the API key for your provider, and that MODEL in Step 3 is a")
    print("valid slug for it (OpenRouter: low/medium/high preset or a real slug")
    print("from https://openrouter.ai/models; Anthropic: e.g. claude-sonnet-5).")

---
## The measurement harness

This cell defines two small helpers we use for the rest of the notebook:

- **`WorkshopSession`** — one agent session with a chosen set of optimizations.
  `.ask(prompt)` runs a turn, `.usage_report()` prints the same output as the
  `/usage` command in the real terminal app.
- **`reset_playground()`** — a clean scratch folder the agent may create and
  edit files in (its tools act on the current directory).

You don't need to read the code to do the workshop — but it's short and it's
exactly how the project's own benchmark runner drives the agent.

In [ ]:
import os, shutil, time
from pathlib import Path

from coding_agent.config import Config
from coding_agent.metrics.pricing import PricingTable
from coding_agent.metrics.usage import UsageTracker
from coding_agent.agent.factory import build_agent
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
from coding_agent.optimizations.registry import OptimizationRegistry
from coding_agent.commands.usage_command import UsageCommand
from coding_agent.models_config import load_catalog_metadata, read_models_yaml

# ANSI colors so the output is easy to read (Jupyter/Colab render these). Same
# scheme as the terminal app: the agent's answer is GREEN, your prompt WHITE,
# and every auxiliary line (per-turn summary, tool calls, /usage, /metrics,
# /cache, warnings) YELLOW; errors RED.
_GREEN, _WHITE, _YELLOW, _RED, _RESET = "\033[32m", "\033[37m", "\033[33m", "\033[31m", "\033[0m"

def _c(text, code):
    """Wrap text in an ANSI color for the notebook's rendered output."""
    return f"{code}{text}{_RESET}"

PLAYGROUND = Path(BASE_DIR) / "playground"

def reset_playground():
    """Wipe and recreate the agent's scratch directory, then move into it."""
    if PLAYGROUND.exists():
        shutil.rmtree(PLAYGROUND)
    PLAYGROUND.mkdir(parents=True, exist_ok=True)
    os.chdir(PLAYGROUND)

class WorkshopSession:
    """An agent session with a chosen set of optimizations enabled."""

    def __init__(self, optimizations=None, enforce_cost_cap=False):
        self.enabled = list(optimizations or [])
        self.config = Config.from_env()
        self.pricing = PricingTable.load()
        self.pricing.require(self.config.model)
        self.usage = UsageTracker()

        bundle = OptimizationRegistry(AVAILABLE_OPTIMIZATIONS).resolve(self.enabled)
        self.agent = build_agent(
            self.config, self.usage, bundle,
            pricing=self.pricing if enforce_cost_cap else None,
        )

        self.configured_models = None
        self.routing_tracker = None
        if "hybrid-routing" in self.enabled:
            from coding_agent.optimizations import hybrid_routing
            from coding_agent.optimizations.routing.tiers import load_tiers
            tiers = load_tiers(provider=self.config.provider)
            for t in tiers:
                self.pricing.require(t.model or self.config.model)
            self.configured_models = [t.model or self.config.model for t in tiers]
            self.routing_tracker = hybrid_routing.get_tracker()
            for w in hybrid_routing.get_warnings():
                print(_c(f"warning> {w}", _YELLOW))

        # Cache-friendly construction records into its own shared tracker,
        # captured here per-session exactly like the routing tracker above.
        self.cache_tracker = None
        if "cache-friendly-prompts" in self.enabled:
            from coding_agent.optimizations import cache_friendly
            self.cache_tracker = cache_friendly.get_tracker()

        self.loop_guard_tracker = None
        if "loop-guard" in self.enabled:
            from coding_agent.optimizations import loop_guard
            self.loop_guard_tracker = loop_guard.get_tracker()

        self.context_tracker = None
        if "context-window" in self.enabled:
            from coding_agent.optimizations import context_window
            self.context_tracker = context_window.get_tracker()

        # The three prompt-optimization techniques (Optimization 2a) each
        # record into their own shared tracker, same pattern as above.
        self.tool_filter_tracker = None
        if "tool-filtering" in self.enabled:
            from coding_agent.optimizations import tool_filtering
            self.tool_filter_tracker = tool_filtering.get_tracker()

        self.compression_tracker = None
        if "prompt-compression" in self.enabled:
            from coding_agent.optimizations import prompt_compression
            self.compression_tracker = prompt_compression.get_tracker()

        self.dedup_tracker = None
        if "deduplication" in self.enabled:
            from coding_agent.optimizations import deduplication
            self.dedup_tracker = deduplication.get_tracker()

        self._meta = load_catalog_metadata(read_models_yaml())

    def ask(self, prompt, show_tools=True):
        """Run one turn: print the tool calls and the final answer."""
        PLAYGROUND.mkdir(parents=True, exist_ok=True)
        os.chdir(PLAYGROUND)
        start = time.perf_counter()

        def on_tool(name, tool_input):
            if show_tools:
                print(_c(f"  [tool] {name}({tool_input})", _YELLOW))

        print(_c(f"you> {prompt}", _WHITE))
        answer = self.agent.run_turn(prompt, on_tool_call=on_tool)
        ms = (time.perf_counter() - start) * 1000
        print(_c(f"\nagent> {answer}", _GREEN))
        summary = (f"  \u21b3 {self.config.model} \u00b7 {self.usage.llm_calls} LLM calls "
                   f"(session) \u00b7 {self.usage.total.total_tokens:,} tokens (session) "
                   f"\u00b7 {ms:.0f}ms this turn")
        print(_c(summary, _YELLOW) + "\n")
        return answer

    def usage_report(self):
        """Print the same thing the /usage terminal command prints."""
        cmd = UsageCommand(
            tracker=self.usage, pricing=self.pricing, config=self.config,
            enabled_optimizations=self.enabled,
            configured_models=self.configured_models,
            model_metadata=self._meta,
            cost_cap_usd=self.config.session_cost_cap_usd,
        )
        print(_c(cmd.run(), _YELLOW))

    def routing_report(self):
        """Print the /metrics command output (only meaningful with routing)."""
        if not self.routing_tracker:
            print("Routing is not enabled for this session.")
            return
        from coding_agent.commands.metrics_command import RoutingMetricsCommand
        report = RoutingMetricsCommand(tracker=self.routing_tracker, pricing=self.pricing).run()
        print(_c(report, _YELLOW))

    def cache_report(self):
        """Print the /cache command output (only with cache-friendly-prompts)."""
        if not self.cache_tracker:
            print("Cache-friendly prompt construction is not enabled for this session.")
            return
        from coding_agent.commands.cache_command import PromptCacheCommand
        print(_c(PromptCacheCommand(tracker=self.cache_tracker).run(), _YELLOW))

    def loop_guard_report(self):
        """Print the /loopguard command output (only with loop-guard)."""
        if not self.loop_guard_tracker:
            print("Loop guard is not enabled for this session.")
            return
        from coding_agent.commands.loop_guard_command import LoopGuardCommand
        print(_c(LoopGuardCommand(tracker=self.loop_guard_tracker).run(), _YELLOW))

    def context_report(self):
        """Print the /context command output (only with context-window)."""
        if not self.context_tracker:
            print("Context window optimization is not enabled for this session.")
            return
        from coding_agent.commands.context_command import ContextWindowCommand
        print(_c(ContextWindowCommand(tracker=self.context_tracker).run(), _YELLOW))

    def tool_filter_report(self):
        """Print the /toolfilter command output (only with tool-filtering)."""
        if not self.tool_filter_tracker:
            print("Tool filtering is not enabled for this session.")
            return
        from coding_agent.commands.tool_filter_command import ToolFilterCommand
        print(_c(ToolFilterCommand(tracker=self.tool_filter_tracker).run(), _YELLOW))

    def compression_report(self):
        """Print the /compression command output (only with prompt-compression)."""
        if not self.compression_tracker:
            print("Prompt compression is not enabled for this session.")
            return
        from coding_agent.commands.compression_command import CompressionCommand
        print(_c(CompressionCommand(tracker=self.compression_tracker).run(), _YELLOW))

    def dedup_report(self):
        """Print the /dedup command output (only with deduplication)."""
        if not self.dedup_tracker:
            print("Deduplication is not enabled for this session.")
            return
        from coding_agent.commands.dedup_command import DedupCommand
        print(_c(DedupCommand(tracker=self.dedup_tracker).run(), _YELLOW))

    def metrics(self, label=None):
        """The measured numbers for this session, as a plain dict."""
        total = self.usage.total
        cost = sum(self.pricing.cost_for(u, m) for m, u in self.usage.by_model.items())
        return {
            "scenario": label or (", ".join(self.enabled) or "base"),
            "optimizations": ", ".join(self.enabled) or "none",
            "llm_calls": self.usage.llm_calls,
            "tool_calls": self.usage.tool_calls,
            "input_tokens": total.input_tokens,
            "output_tokens": total.output_tokens,
            "total_tokens": total.total_tokens,
            "cost_usd": round(cost, 6),
        }

print("Harness ready: WorkshopSession, reset_playground().")


In [ ]:
def run_scenario(label, optimizations, prompts, show_tools=False, setup=None):
    """Run the SAME prompts through a fresh agent with the chosen optimizations.

    setup, if given, runs right after the playground is wiped and before the
    session starts - e.g. seeding a fixture file every scenario should see.
    """
    reset_playground()
    if setup is not None:
        setup()
    session = WorkshopSession(optimizations=optimizations)
    opt_label = ", ".join(optimizations) or "none"
    print(_c(f"=== Scenario: {label}  (optimizations: {opt_label}) ===", _YELLOW))
    for p in prompts:
        session.ask(p, show_tools=show_tools)
    m = session.metrics(label)
    print(_c(f"--- {label}: {m['total_tokens']:,} tokens \u00b7 ${m['cost_usd']:.4f} "
             f"\u00b7 {m['llm_calls']} LLM calls ---", _YELLOW) + "\n")
    return {"session": session, "metrics": m}

def _pct(base, new):
    return 0.0 if base == 0 else (base - new) / base * 100.0

def compare(baseline, *others):
    """Show baseline vs one or more optimized runs, with % saved."""
    rows = [baseline["metrics"]] + [o["metrics"] for o in others]
    base = baseline["metrics"]
    try:
        import pandas as pd
        from IPython.display import display
        df = pd.DataFrame(rows)
        df["tokens_saved_%"] = df["total_tokens"].apply(
            lambda t: round(_pct(base["total_tokens"], t), 1))
        df["cost_saved_%"] = df["cost_usd"].apply(
            lambda c: round(_pct(base["cost_usd"], c), 1))
        cols = ["scenario", "llm_calls", "tool_calls", "total_tokens",
                "tokens_saved_%", "cost_usd", "cost_saved_%"]
        display(df[cols])
        return df
    except Exception:
        print(_c(f"{'scenario':<26}{'tokens':>10}{'saved%':>9}{'cost$':>11}{'saved%':>9}", _YELLOW))
        for r in rows:
            print(_c(f"{r['scenario']:<26}{r['total_tokens']:>10,}"
                     f"{_pct(base['total_tokens'], r['total_tokens']):>8.1f}%"
                     f"{r['cost_usd']:>11.4f}"
                     f"{_pct(base['cost_usd'], r['cost_usd']):>8.1f}%", _YELLOW))

print("Ready: run_scenario(...), compare(...).")


---
## Meet the base agent

No optimizations yet. Watch the `[tool]` lines — the agent really reads and
writes files and runs commands; it doesn't just *describe* what to do. Edit the
prompt and re-run to experiment.

In [ ]:
reset_playground()
base_demo = WorkshopSession()   # no optimizations
base_demo.ask("Create hello.py that prints 'Hello, workshop!', then show me its contents.")

### The `/usage` command

In the terminal app you'd type `/usage` to see tokens, cost, and which
optimizations are on. Here's the exact same report for the session above.

In [ ]:
base_demo.usage_report()

---
## The shared prompt set

To compare fairly, every configuration runs the **same fixed conversation**. It
builds a small module across several turns — each turn depends on the previous
ones. That growing history is exactly what *conversation summarization*
compresses, the routine edits are what *model routing* sends to a cheaper
model, the verification turn re-reads unchanged files (what *deduplication*
collapses), and the read-only turns need no write/edit/bash tools (what
*tool filtering* withholds).

> The task never changes, only the optimization does — so any difference in the
> numbers is caused by the optimization, not by a different prompt.

In [ ]:
DEMO_PROMPTS = [
    "Create a file calculator.py with a function add(a, b) that returns their sum. Keep it minimal.",
    "Add a subtract(a, b) function to calculator.py.",
    "Add multiply(a, b) and divide(a, b) to calculator.py. divide must raise ValueError on division by zero.",
    "Create test_calculator.py with pytest tests for all four functions, including the divide-by-zero case.",
    "Read calculator.py and test_calculator.py again in full to double-check them, and confirm the tests cover every function.",
    "List the files you created and give a one-line summary of what calculator.py now contains.",
]
print(f"{len(DEMO_PROMPTS)} prompts ready.")

---
## Baseline — the un-optimized agent

Run the shared prompts with **no** optimizations and record the numbers. Every
later scenario is measured against this. (This calls the model several times, so
give it a moment.)

In [ ]:
baseline = run_scenario("base agent", [], DEMO_PROMPTS)
baseline["session"].usage_report()

---
## Optimization 1 — Conversation summarization

Every turn, the agent resends the **entire** conversation history to the model.
As the chat grows, so do the input tokens on *every* call — you pay again and
again to resend old context.

**Conversation summarization** compresses older messages into a short running
summary once history passes a threshold, keeping only the last few messages
verbatim. Fewer input tokens per call → lower cost, with recent context intact.

It's one of the five *prompt-optimization* techniques (the full family is laid
out in Optimization 2a, next) — the history-compressing member, big enough to
measure on its own first.

*Terminal equivalent:* `uv run coding-agent --enable conversation-summary`

In [ ]:
opt_summary = run_scenario("+ conversation-summary", ["conversation-summary"], DEMO_PROMPTS)
compare(baseline, opt_summary)

---
## Optimization 2 — Prompt optimization & caching

Most of every request is **repeated** each turn: the system prompt, the tool
definitions, and the established conversation get resent on every single call.
You keep paying full price for tokens the model has already seen — many of
which it never needed in the first place.

There are **two levers** here, mirroring the two slides in the deck:

- **2a · Prompt optimization** — send *fewer* tokens: a family of five
  techniques that each remove waste from the prompt itself (three of them
  measured right here, two in their own sections).
- **2b · Cache-friendly prompts** — make the tokens that *must* repeat
  **cheaper to resend**: build the prompt deterministically so its unchanging
  prefix is byte-identical every turn — the groundwork a provider-side cache
  needs before it can reuse (and re-bill at a fraction) anything.

### 2a — Prompt optimization: the five techniques

"Prompt optimization" isn't one trick — it's a family of five, each attacking a
different kind of waste in what gets sent. All five are **live in the repo**,
each behind its own flag:

| Technique | What it removes | Flag | Measured |
|-----------|-----------------|------|----------|
| **Context pruning** | irrelevant conversation history (stale bulky tool output) | `context-window` | Optimization 5, below |
| **Conversation summarization** | older messages, replaced by a concise summary | `conversation-summary` | Optimization 1, above |
| **Tool filtering** | tool definitions irrelevant to the current request | `tool-filtering` | **here** |
| **Prompt compression** | long instructions, rewritten into shorter equivalents | `prompt-compression` | **here** |
| **Deduplication** | identical instructions/content repeated in one request | `deduplication` | **here** |

Two of the five are big enough to have their own sections (summarization ran as
Optimization 1; pruning runs as Optimization 5) — they're the *history-policy*
members of the family. The other three all wrap the model call itself, so they
compose freely with anything — we run each one below.

A shared design stance across all five: the removal is **deterministic and
local** (string matching, hashing, hand-tightened rewrites) — never a model
call that spends tokens to save tokens, and never an estimate.

#### Anatomy of a request — what actually gets sent every call

Before measuring anything, look at the raw material. Every single call the
agent makes carries: the **system prompt**, the **tool definitions**, the
**conversation so far**, and the **latest turn**. The repo tags each part by
*volatility* — how often it changes — because that's what decides both what's
worth optimizing (2a) and what's cacheable (2b):

| Layer | Volatility | Prompt-optimization lever |
|-------|:---:|---------------------------|
| System prompt | `stable` | prompt compression (tighter wording) |
| Tool definitions | `stable` | tool filtering + prompt compression |
| Conversation so far | `semi-stable` | summarization · pruning · deduplication |
| Latest message & tool results | `dynamic` | the only genuinely new tokens |

The next cell prints the real thing — the actual system prompt the agent
sends (and its compressed equivalent), a real tool definition, and a sample
conversation run through the repo's own layering pipeline so you can see the
stable → semi-stable → dynamic split in real bytes.

In [ ]:
import json

from coding_agent.system_prompt import SYSTEM_PROMPT, SYSTEM_PROMPT_COMPACT
from coding_agent.tools.registry import ToolRegistry
from coding_agent.tools.read_file import ReadFileTool
from coding_agent.tools.write_file import WriteFileTool
from coding_agent.tools.edit_file import EditFileTool
from coding_agent.tools.bash import BashTool
from coding_agent.tools.list_files import ListFilesTool
from coding_agent.llm.messages import Message, TextPart, ToolResultPart, ToolUsePart
from coding_agent.optimizations.prompt_cache.builder import PromptBuilder

# The exact tool set agent/factory.py builds (bash timeout from the same config).
_tools = ToolRegistry(tools=[
    ReadFileTool(), WriteFileTool(), EditFileTool(),
    BashTool(timeout_seconds=cfg.bash_timeout_seconds), ListFilesTool(),
]).definitions()

print(_c("=== STABLE · the system prompt (verbatim, sent with EVERY call) ===", _YELLOW))
print(SYSTEM_PROMPT)
print(_c(f"--- its hand-tightened equivalent, swapped in by --enable prompt-compression "
         f"({len(SYSTEM_PROMPT)} -> {len(SYSTEM_PROMPT_COMPACT)} chars, same rules) ---", _YELLOW))
print(SYSTEM_PROMPT_COMPACT)

print(_c(f"=== STABLE · tool definitions ({len(_tools)} tools, also sent with EVERY call) ===", _YELLOW))
print(_c(f"one of them, verbatim (tool filtering withholds irrelevant ones per request):", _YELLOW))
print(json.dumps(_tools[0], indent=2))

# A sample mid-task conversation, so the semi-stable/dynamic layers have content.
_sample_conversation = [
    Message(role="user", parts=[TextPart("Create calculator.py with an add function.")]),
    Message(role="assistant", parts=[ToolUsePart(id="c1", name="write_file",
            input={"path": "calculator.py", "content": "def add(a, b):\n    return a + b\n"})]),
    Message(role="user", parts=[ToolResultPart(tool_use_id="c1",
            output="Wrote calculator.py", is_error=False)]),
    Message(role="user", parts=[TextPart("Now add a subtract function.")]),
]

# The repo's own layering pipeline (optimizations/prompt_cache/) — the same
# code --enable cache-friendly-prompts runs on every send.
_built = PromptBuilder().build(system=SYSTEM_PROMPT, messages=_sample_conversation, tools=_tools)

print(_c("=== The layered request, stable -> semi-stable -> dynamic ===", _YELLOW))
for _layer in _built.layers:
    print(f"  {_layer.tier.name:<12} {_layer.name}")
print(_c(
    f"\nbytes: stable {_built.stable_bytes:,} \u00b7 semi-stable {_built.semi_stable_bytes:,} "
    f"\u00b7 dynamic {_built.dynamic_bytes:,} \u00b7 total {_built.total_bytes:,}"
    f"\nstable share of the whole request: {_built.cache_friendly_ratio * 100:.0f}%"
    f"\nstable prefix fingerprint: {_built.stable_hash[:16]}\u2026 (2b's job: keep this identical every send)",
    _YELLOW,
))

#### 2a · Tool filtering — expose only what the request needs

Every tool definition rides along on every send, whether the request needs it
or not. **Tool filtering** scores the latest user message with free, local
keyword heuristics and withholds the *action* tools (`write_file`,
`edit_file`, `bash`) from requests that don't ask for them.

Three safety rules keep it honest — filtering must never break the agent:
1. `read_file`/`list_files` are **never** withheld (the system prompt tells
   the model to explore before acting), nor is any unknown extra tool.
2. A tool already used in the current turn's loop stays exposed.
3. No confident keyword match → **all** tools kept. When uncertain, spend the
   tokens rather than risk correctness.

Watch the read-only turns (the double-check and list-files prompts): those are
where definitions get withheld. The `/toolfilter` report after the run shows
exactly which. *Terminal equivalent:* `uv run coding-agent --enable tool-filtering`

In [ ]:
opt_tool_filter = run_scenario("+ tool-filtering", ["tool-filtering"], DEMO_PROMPTS)
compare(baseline, opt_tool_filter)
opt_tool_filter["session"].tool_filter_report()

#### 2a · Prompt compression — same instructions, fewer words

The system prompt and every tool description are resent on **every single
call** — so a tighter wording of the same rules is a pure win that compounds
with volume and costs nothing at runtime.

The compression is **hand-written and deterministic**, not a model rewriting
text on the fly: `SYSTEM_PROMPT_COMPACT` (you saw both versions in the anatomy
cell above) and compact tool descriptions were tightened once by a human,
reviewed to mean the same thing, and swapped in verbatim at the send boundary.
A runtime "compressor" model would spend tokens to save tokens and could
silently change meaning — this can't. Two details that make it safe to stack:
anything another optimization appends to the system prompt (like 2b's skills
menu) passes through untouched, and unknown tools keep their original
descriptions.

Unlike the other techniques, this one fires on *every* send — expect a steady
input-token drop across the board, not a few big wins. *Terminal equivalent:*
`uv run coding-agent --enable prompt-compression`

In [ ]:
opt_compression = run_scenario("+ prompt-compression", ["prompt-compression"], DEMO_PROMPTS)
compare(baseline, opt_compression)
opt_compression["session"].compression_report()

#### 2a · Deduplication — never resend what the model already saw

An agent conversation naturally accumulates **exact duplicates**: the model
re-reads the same file across turns, re-runs the same command, or the user
pastes the same instructions twice. Every copy is billed again on every
subsequent call — for bytes the model has already seen verbatim.

**Deduplication** keeps the *first* occurrence of any large block intact and
replaces later exact duplicates with a short marker pointing back at it:
`[duplicate removed: identical to an earlier tool_result above, 812 chars -
the first copy is still in context]`. Only exact matches count (never
similarity — meaning is never changed), only blocks past a size threshold
qualify, no message is ever removed, and the agent's own memory of the
conversation is untouched — only what gets *sent* per call shrinks.

The double-check turn in `DEMO_PROMPTS` re-reads files that haven't changed
since the previous read — that's where the markers appear. *Terminal
equivalent:* `uv run coding-agent --enable deduplication`

In [ ]:
opt_dedup = run_scenario("+ deduplication", ["deduplication"], DEMO_PROMPTS)
compare(baseline, opt_dedup)
opt_dedup["session"].dedup_report()

### 2b — Prompt caching & cache-friendly prompts

After 2a has removed the waste, a hard core of the prompt still *must* repeat
every turn — the system prompt, the tool definitions, the established
conversation. The second lever makes those repeats **cheaper** instead of
smaller: mark the unchanging prefix as cacheable at the API boundary, and on a
cache hit the provider re-bills that prefix at roughly a tenth of the price —
only the genuinely new tokens pay the full rate.

> Caching doesn't cut the *number* of tokens — it cuts the *price* of the
> repeated ones, so a token gauge barely moves while cost growth flattens.
> Provider-side billing is the **groundwork-in-progress** half; the half
> that's **live in the repo** is the construction it depends on, below.

**Cache-friendly construction.** A cache can only reuse a prefix that is
**byte-identical** every turn, so the prompt is built deterministically —
stable parts first, churn last — with tools sorted, JSON canonicalized, and
whitespace normalized. The stable prefix then hashes the same on every send,
giving a cache (provider-side, or a future adapter) something byte-stable to
reuse.

**Layer by volatility — stable first, churn last.** You saw the real layers in
the anatomy cell above; the build *order* is the optimization:

| Layer | Volatility | What's in it |
|-------|:---:|--------------|
| **System prompt · tool definitions** | `stable` | Never change within a session — serialized once, reused every send. |
| **Repository metadata** | `stable` | Coding guidelines, project facts — constant all session. |
| **Conversation summary · active files** | `semi-stable` | Change occasionally as the task moves along. |
| **Latest message · tool results** | `dynamic` | New on every single send. |

Always **stable → semi-stable → dynamic**, never interleaved. One stray reorder
of a tool, or a timestamp in the stable section, moves the prefix hash and
defeats the cache.

> It doesn't cut tokens by itself — it makes them *cacheable*. The win shows up
> as a stable prefix hash that never moves and a rising **reuse %**, both
> measured in real bytes, never estimated.

*Terminal equivalent:* `uv run coding-agent --enable cache-friendly-prompts`

In [ ]:
opt_cache = run_scenario("+ cache-friendly-prompts", ["cache-friendly-prompts"], DEMO_PROMPTS)
compare(baseline, opt_cache)

### The `/cache` command

With cache-friendly construction on, `/cache` reports the stable-prefix hash
(it should stay identical across every send), how much of each request was a
reusable prefix, and the structurally cacheable share of the prompt. It also
breaks the latest prompt into the three layers — **stable · semi-stable ·
dynamic** — in real bytes. The token count is the provider's real number; the
byte-based figures are deterministic, not estimated.

In [ ]:
opt_cache["session"].cache_report()

---
## Optimization 3 — Model routing (hybrid routing)

Not every request needs your most capable (most expensive) model. **Hybrid
routing** scores each request's difficulty (free, local) and sends easy/routine
work to a **cheaper** model, escalating to a stronger tier only when a quality
gate flags the cheap answer.

The base agent pays the high-tier price for *everything*; routing pays cheap-tier
prices for the routine edits. The ladder is defined in `models.yaml`
(`routing.tiers`) — a data edit, no code change. The three rungs are the
standardized OpenRouter presets:

| Tier | Difficulty ceiling | Model | Input $/M | Output $/M | Best for |
|------|:---:|-------|:---:|:---:|----------|
| **low** · cheap | ≤ 0.45 | `google/gemma-3.4b` | $0.05 | $0.10 | routine edits, boilerplate |
| **medium** · mid | ≤ 0.75 | `qwen/qwen3.7-flash` | $0.03 | $0.13 | reasoning, debugging, tool use |
| **high** | ≤ 1.0 | `deepseek/deepseek-v4-flash-0731` | $0.08 | $0.252 | architecture, hard algorithms |

A request is scored `0.0`–`1.0` and starts at the first tier whose ceiling
covers it, so hard requests skip the cheap rungs entirely.

> This ladder is OpenRouter-based, so this optimization needs an
> **OpenRouter** key. If you configured Anthropic in Step 3, skip this cell
> (its tiers would be unavailable) — the other optimizations are
> provider-agnostic and work on either provider.

*Terminal equivalent:* `uv run coding-agent --enable hybrid-routing`

In [ ]:
opt_routing = run_scenario("+ hybrid-routing", ["hybrid-routing"], DEMO_PROMPTS)
compare(baseline, opt_routing)

### The `/metrics` command

With routing on, `/metrics` breaks down which tier answered, how often the
cheap model was enough, and how often it escalated.

In [ ]:
opt_routing["session"].routing_report()

---
## Optimization 4 — Agent loop prevention

`system_prompt.py` already tells the model, in plain English: *"do not repeat
the exact same call expecting a different result."* That's today's
**prompt-side** loop prevention, and it works most of the time — but "most of
the time" isn't a cost cap.

**`loop-guard`** is the code-side backstop: it watches the tail of the
conversation for the same tool call failing with the same error, repeatedly.
- After a couple of identical failures in a row, it injects a corrective
  nudge before the next real model call.
- After a few more, it stops calling the model entirely and returns a clean
  "loop detected, stopping" answer instead — no further API calls, so no
  further cost, for a call that wasn't making progress anyway.

Run it against the same cooperative `DEMO_PROMPTS` first. **Expect zero
nudges and zero halts here** — a well-behaved model building a calculator
module shouldn't get stuck, and a workshop that faked a loop just to show a
chart would defeat the point of measuring anything. Zero is the correct,
honest result on this prompt set.

*Terminal equivalent:* `uv run coding-agent --enable loop-guard`


In [ ]:
opt_loop_guard = run_scenario("+ loop-guard", ["loop-guard"], DEMO_PROMPTS)
compare(baseline, opt_loop_guard)


### The `/loopguard` command

Total sends this session, how many were nudged, how many were halted, and
the current repeat streak. No estimated "tokens saved by halting" figure —
this project only ever reports real, measured numbers, and guessing what a
skipped call *would have* cost would break that.


In [ ]:
opt_loop_guard["session"].loop_guard_report()


### Trying to actually trip it

Getting a real model to genuinely get stuck on demand is inherently
unreliable — models are usually smart enough not to loop, which is a good
thing, not a demo inconvenience. This prompt nudges it toward retrying a
command that can't succeed as written, run once with `loop-guard` and once
without. Treat the comparison as illustrative, not a guaranteed trip — the
point is that the guard is there as a catastrophe cap *if* a session ever
does get stuck, not that this specific prompt reliably proves it every run.


In [ ]:
STRESS_PROMPT = (
    "Run the shell command `python3 -m nonexistent_module_xyz` and keep "
    "trying variations of the command until one of them works. Don't stop "
    "and ask me anything - keep attempting fixes yourself."
)

without_guard = run_scenario("stress, no loop-guard", [], [STRESS_PROMPT])
with_guard = run_scenario("stress, + loop-guard", ["loop-guard"], [STRESS_PROMPT])
compare(without_guard, with_guard)
with_guard["session"].loop_guard_report()


---
## Optimization 5 — Context window optimization

The *context-pruning* member of the prompt-optimization family from 2a — two
mechanisms, both about *relevance* rather than compression (contrast with
conversation-summary, which compresses old turns into prose):

1. **Prune stale bulky tool output.** Once a tool result (e.g. a big
   `read_file` dump) falls outside the most recent few messages, its output
   gets replaced with a short, specific placeholder instead of being resent
   on every later call — "\[pruned: read\_file output for 'x.py', 812
   chars\]", never a vague "something was removed."
2. **Skills, loaded on demand.** The system prompt carries only a short menu
   of skill names + one-line descriptions; the full guidance for a skill only
   enters context when the model actually calls `load_skill(name)`. The
   `pytest-conventions` skill shipped with this optimization is written to
   match the "Create test\_calculator.py..." prompt already in
   `DEMO_PROMPTS` — watch the `[tool]` lines below for a `load_skill` call
   when that turn runs.

*Terminal equivalent:* `uv run coding-agent --enable context-window`


In [ ]:
opt_context = run_scenario("+ context-window", ["context-window"], DEMO_PROMPTS, show_tools=True)
compare(baseline, opt_context)


### The `/context` command

Tool outputs pruned and their total size removed (a deterministic character
count, not a token estimate), plus exactly which skills got loaded this
session — so you can see whether `load_skill("pytest-conventions")` actually
fired on the test-writing turn.


In [ ]:
opt_context["session"].context_report()


### A demo that actually gives pruning something to prune

The comparison above came out **worse**, not better — `context-window` added
~19% more tokens on `DEMO_PROMPTS` than the baseline. That's a real result,
not a fluke, and it's worth understanding why before trusting any number this
notebook prints.

`context-window` bundles two mechanisms with opposite cost signs:

- **Pruning** only *removes* something when a bulky tool *result* sits far
  enough back in history to fall outside `keep_recent_messages` (6 messages)
  before the conversation ends. `DEMO_PROMPTS` builds a deliberately tiny
  `calculator.py` — and `write_file`/`edit_file` return one-line confirmations
  ("Wrote 41 characters..."), not the file content, so there's rarely
  anything bulky to prune in the first place. The one turn that *does* read a
  file in full ("read again to double-check") lands near the *end* of the
  conversation — still inside the recent window, so it's never old enough to
  prune before the run finishes.
- **Skill loading** adds a fixed cost to *every single call* (the skills menu
  in the system prompt + the `load_skill` tool definition), whether or not
  pruning ever fires.

A fixed cost with nothing to offset it is a net loss — exactly what the table
above shows.

Below is a session shaped the way pruning is actually meant for: **read one
large, rarely-needed file once, early — then do several small, unrelated
turns.** That's the "skim the legacy module, then go do local work elsewhere"
pattern real sessions have all the time. The big read should age past the
recent-messages window and get pruned on every later call, while the small
edits cost next to nothing either way — so pruning finally has real, bulky,
stale content to remove.

*Terminal equivalent:* same flag, `uv run coding-agent --enable
context-window`, just pointed at a repo with an actual big file to skim.

In [ ]:
LEGACY_UTILS_SRC = '''"""legacy_utils.py -- assorted helpers accumulated over the project\'s early
days. Nobody owns this file anymore; a function gets added here whenever
someone needs a quick helper and doesn\'t want to start a new module."""


def slugify(text):
    """Turn text into a lowercase, hyphen-separated, URL-safe slug."""
    cleaned = "".join(ch if ch.isalnum() or ch.isspace() else "" for ch in text)
    return "-".join(cleaned.lower().split())


def truncate(text, max_length=80, suffix="..."):
    """Shorten text to max_length characters, appending suffix if cut."""
    if len(text) <= max_length:
        return text
    return text[: max_length - len(suffix)] + suffix


def chunk_list(items, size):
    """Split items into consecutive chunks of at most `size` elements."""
    return [items[i : i + size] for i in range(0, len(items), size)]


def flatten(nested):
    """Flatten one level of nested lists into a single list."""
    result = []
    for item in nested:
        if isinstance(item, list):
            result.extend(item)
        else:
            result.append(item)
    return result


def dedupe_preserve_order(items):
    """Remove duplicates from items while keeping first-seen order."""
    seen = set()
    result = []
    for item in items:
        if item not in seen:
            seen.add(item)
            result.append(item)
    return result


def is_palindrome(text):
    """True if text reads the same forwards/backwards, ignoring case and spaces."""
    cleaned = "".join(ch.lower() for ch in text if ch.isalnum())
    return cleaned == cleaned[::-1]


def parse_csv_line(line, delimiter=","):
    """Split one CSV line into fields, stripping surrounding whitespace."""
    return [field.strip() for field in line.split(delimiter)]


def merge_dicts(*dicts):
    """Merge dicts left to right; later dicts override earlier keys."""
    merged = {}
    for d in dicts:
        merged.update(d)
    return merged


def clamp(value, low, high):
    """Clamp value into the inclusive [low, high] range."""
    return max(low, min(value, high))


def retry_count_from_env(env_value, default=3):
    """Parse an integer retry count from an env var string, or fall back."""
    try:
        return max(0, int(env_value))
    except (TypeError, ValueError):
        return default


def humanize_bytes(num_bytes):
    """Format a byte count as a short human-readable string, e.g. \'3.2MB\'."""
    for unit in ("B", "KB", "MB", "GB", "TB"):
        if num_bytes < 1024:
            return f"{num_bytes:.1f}{unit}" if unit != "B" else f"{num_bytes}{unit}"
        num_bytes /= 1024
    return f"{num_bytes:.1f}PB"


def days_between(date_a, date_b):
    """Return the absolute number of days between two datetime.date objects."""
    return abs((date_b - date_a).days)


def normalize_whitespace(text):
    """Collapse any run of whitespace in text into a single space."""
    return " ".join(text.split())


def deprecated_swap_case(text):
    """Old helper that swaps the case of every character. Unused since the
    2019 rewrite of the formatting pipeline -- kept here in case something
    still imports it."""
    return text.swapcase()
'''

def seed_legacy_file():
    """Write the fixture above into the (already-reset) playground."""
    (PLAYGROUND / "legacy_utils.py").write_text(LEGACY_UTILS_SRC)

print(f"Fixture ready: legacy_utils.py, {len(LEGACY_UTILS_SRC):,} chars.")

CONTEXT_PRUNING_DEMO_PROMPTS = [
    "Read legacy_utils.py in full and tell me in 2-3 sentences what kinds of "
    "helpers it contains.",
    "Create a file shapes.py with a function area_circle(radius) that "
    "returns the area of a circle. Keep it minimal.",
    "Add a function area_rectangle(width, height) to shapes.py.",
    "Add a function area_triangle(base, height) to shapes.py.",
    "List the files in the current directory.",
    "Based only on what you already read earlier in this conversation about "
    "legacy_utils.py - do not call read_file again - name one function from "
    "it whose name suggests it might be dead code, in one sentence.",
    "Give a one-line summary of what shapes.py now contains.",
]
print(f"{len(CONTEXT_PRUNING_DEMO_PROMPTS)} prompts ready.")

In [ ]:
baseline_ctx = run_scenario(
    "base agent (pruning demo)", [], CONTEXT_PRUNING_DEMO_PROMPTS,
    setup=seed_legacy_file,
)
opt_context2 = run_scenario(
    "+ context-window (pruning demo)", ["context-window"],
    CONTEXT_PRUNING_DEMO_PROMPTS, show_tools=True, setup=seed_legacy_file,
)
compare(baseline_ctx, opt_context2)

In [ ]:
opt_context2["session"].context_report()

---
## Stack them — combined optimizations

Optimizations **compose** as long as they don't both claim the same hook.
`context-window` controls *what history is sent* (the one `history_policy`
below); `hybrid-routing`, `loop-guard`, `tool-filtering`,
`prompt-compression`, and `deduplication` all wrap the model call itself, and
wrappers **chain** — every one of them applies, in order. Six flags, no
conflict. (`context-window` and `conversation-summary` can't combine — both
are a `history_policy`, and only one policy can own history at a time;
enabling both raises `ConflictingOptimizationsError` on purpose rather than
silently picking a winner.) This is usually where the biggest savings show up.

*Terminal equivalent:*
`uv run coding-agent --enable hybrid-routing,loop-guard,context-window,tool-filtering,prompt-compression,deduplication`

In [ ]:
opt_both = run_scenario(
    "+ six stacked",
    ["hybrid-routing", "loop-guard", "context-window",
     "tool-filtering", "prompt-compression", "deduplication"],
    DEMO_PROMPTS,
)
compare(baseline, opt_summary, opt_tool_filter, opt_compression, opt_dedup,
        opt_routing, opt_cache, opt_loop_guard, opt_context, opt_both)


---
## Scoreboard

The whole story in two charts: total tokens and estimated cost, baseline vs each
optimization vs both.

In [ ]:
runs = [baseline, opt_summary, opt_tool_filter, opt_compression, opt_dedup,
        opt_routing, opt_cache, opt_loop_guard, opt_context, opt_both]
try:
    import matplotlib.pyplot as plt
    labels = [r["metrics"]["scenario"] for r in runs]
    tokens = [r["metrics"]["total_tokens"] for r in runs]
    costs = [r["metrics"]["cost_usd"] for r in runs]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
    ax1.bar(labels, tokens); ax1.set_title("Total tokens (lower is better)")
    ax1.tick_params(axis="x", rotation=30)
    ax2.bar(labels, costs); ax2.set_title("Estimated cost, USD (lower is better)")
    ax2.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()
except Exception as e:
    print("Chart skipped:", e)
    for r in runs:
        m = r["metrics"]
        print(f"{m['scenario']:<26} {m['total_tokens']:>8,} tokens  ${m['cost_usd']:.4f}")


---
## Coming soon — more optimizations

`loop-guard`, `context-window`, and the three 2a techniques
(`tool-filtering`, `prompt-compression`, `deduplication`) all started here as
"coming soon" — they're now real, measurable optimizations on the same
plug-in system. What's still genuinely unbuilt:

- **Provider-side prompt caching** — the cache-friendly construction above
  already builds a byte-stable prefix; the remaining step is marking it
  cacheable at the provider boundary via the adapter seam, so the repeated
  prefix is billed at a fraction.

When it ships, it shows up in the list below automatically. Add its name to a
`run_scenario` cell and re-run `compare(...)` to measure it.


In [ ]:
from coding_agent.optimizations.available import AVAILABLE_OPTIMIZATIONS
print("Optimizations available right now:")
for name in AVAILABLE_OPTIMIZATIONS:
    print("  \u2022", name)

---
## Your turn — free play

Enable any combination of the available optimizations and give the agent your
own task. Try the same prompt with and without an optimization and compare
`usage_report()` output.

In [ ]:
reset_playground()
my_session = WorkshopSession(optimizations=[])   # e.g. ["conversation-summary", "hybrid-routing"]
my_session.ask("Write a Python function that reverses a string without slicing, and show it to me.")
my_session.usage_report()

---
## Appendix — running the real terminal agent

Everything above drives the **same** agent you'd run in a terminal — just
in-process so we can measure it cleanly. To run the actual REPL locally
(outside Colab):

```bash
git clone https://github.com/shrijayan/coding-agent.git
cd coding-agent
uv sync
cp .env.example .env          # add your OPENROUTER_API_KEY or ANTHROPIC_API_KEY
uv run coding-agent --enable conversation-summary,hybrid-routing
```

Inside the REPL, type `/usage` (and `/metrics` with routing enabled) to see the
same numbers this notebook computes. To measure correctness as well as cost,
`uv run coding-agent --benchmark --enable <name>` runs a fixed suite of real
coding tasks.